In [ ]:
import os

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy.spatial import cKDTree
from sklearn.manifold import Isomap

from pmhclib.components.pmhc import pMHCII
from pmhclib.modeling import Ensemble
from pmhclib.representations.structure import TorsionVector

pio.templates.default = "plotly_white"

In [ ]:
pdb_ids = ["1FYT", "3QIB", "4OZF", "3MBE", "4P2O", "6BGA"]

template_dir = "../data/mhcii_templates/"
init_model_dir = "../data/mhcii_models/pandora2/"

templates = dict()
models = dict()

for pdb_id in pdb_ids:
    templates[pdb_id] = pMHCII(os.path.join(template_dir, f"{pdb_id}.pdb"), id=f"{pdb_id}__template")
    models[pdb_id] = pMHCII(os.path.join(init_model_dir, f"{pdb_id}.pdb"), id=f"{pdb_id}__model")

In [ ]:
ensembles_dir = "../data/mhcii_ensembles/"
sampling_methods = ["amber", "annealing", "pandora2"]
colors = ["blue", "red", "green"]

In [ ]:
df = None

for pdb_id in pdb_ids:
    for sampling_method in sampling_methods:
        ensemble_dir = os.path.join(pdb_id, sampling_method)
        csv_fp = os.path.join(ensembles_dir, ensemble_dir, "stats.csv")
        if not os.path.exists(csv_fp):
            continue
        ensemble_df = pd.read_csv(csv_fp, index_col="model_id")
        ensemble_df.drop("pdb_fp", axis=1, inplace=True)
        ensemble_df["pdb_id"] = pdb_id
        ensemble_df["sampling_method"] = sampling_method
        if df is None:
            df = ensemble_df
        else:
            df = pd.concat([df, ensemble_df])

df

In [ ]:
metrics = ["rmsd", "dockq", "rosetta"]

In [ ]:
for metric in metrics:
    fig = go.Figure()
    for i, pdb_id in enumerate(pdb_ids):
        pdb_df = df[df.pdb_id == pdb_id]
        for sampling_method, color in zip(sampling_methods, colors):
            ens_df = pdb_df[pdb_df.sampling_method == sampling_method]
            fig.add_trace(
                go.Box(
                    y=ens_df[metric].values,
                    name=pdb_id,
                    offsetgroup=sampling_method,
                    legendgroup=sampling_method,
                    showlegend=(i == 0),
                    line=dict(color=color),
                    marker=dict(color=color)
                )
            )
    fig.update_layout(title=metric, boxmode="group")
    fig.show()

In [ ]:
ensembles = dict()
fn_patterns = ["*_frame_*.pdb", "conf_*.pdb", "*.BL*.pdb"]

for pdb_id in pdb_ids:
    ensembles[pdb_id] = dict()
    for sampling_method, fn_pattern in zip(sampling_methods, fn_patterns):
        ensemble_dir = os.path.join(ensembles_dir, pdb_id, sampling_method)
        print(ensemble_dir)
        ensemble = Ensemble.from_pdb_dir(
            id=f"{pdb_id}__{sampling_method}", pdb_dir=ensemble_dir,
            fn_pattern=fn_pattern
        )
        ensembles[pdb_id][sampling_method] = ensemble
        print(len(ensemble.ids))

In [ ]:
featurizer = TorsionVector(angles=["psi", "phi"])

def symmetric_avg_min_distance(X, y, group_a, group_b=0):
    """
    Compute symmetric average minimum distance between group_a and group_b.

    X : (m, n) numpy array of points
    y : (m,) array of labels
    group_a : int (e.g., 1 or 2)
    group_b : int (default 0)
    """
    Xa = X[y == group_a]
    Xb = X[y == group_b]

    if len(Xa) == 0 or len(Xb) == 0:
        return np.nan  # or raise error

    tree_a = cKDTree(Xa)
    tree_b = cKDTree(Xb)

    # distances from A → nearest in B
    dists_a_to_b, _ = tree_b.query(Xa, k=1)

    # distances from B → nearest in A
    dists_b_to_a, _ = tree_a.query(Xb, k=1)

    return 0.5 * (dists_a_to_b.mean() + dists_b_to_a.mean())

def compute_for_groups(X, y):
    results = {}
    for g in [1, 2]:
        results[g] = symmetric_avg_min_distance(X, y, g, group_b=0)
    return results

for pdb_id in pdb_ids:
    X = None
    y = np.array([])
    conf_ids = list()
    for i, (_, ensemble) in enumerate(ensembles[pdb_id].items()):
        if X is None:
            X = ensemble.apply_representation(featurizer, selection="chain P")
        else:
            X = np.concatenate((X, ensemble.apply_representation(featurizer, selection="chain P")))
        y = np.concatenate((y, [i] * len(ensemble.ids)))
        conf_ids.extend(ensemble.ids)
    y = y.astype(int)
    dr = Isomap(n_components=2)
    X_p = dr.fit_transform(X)
    fig = px.scatter(
        x=X_p[:, 0], y=X_p[:, 1], hover_name=conf_ids,
        color=[sampling_methods[i] for i in y]
    )
    fig.show()
    distances = compute_for_groups(X_p, y)
    print(distances[1], distances[2])